# Omer Gamie 的第 1 周 · 第 2 天练习

[Omer Gamie](https://github.com/OmerGamie) — 初级 AI/LLM 工程师

## 练习目标（理念）

用 **Ollama**（OpenAI 兼容接口）抓取网页并生成**简短、带点吐槽风格**的摘要：

- **输入**：网站 URL
- **流程**：`fetch_website_contents` 抓正文 → 拼 system/user messages → 本地 `llama3.2` 总结
- **输出**：Markdown 摘要（不要包在代码块里）

这是 Day 1 网页摘要器换成**本地模型**的变体：同一套 Chat Completions 写法，后端指向 `localhost:11434/v1`。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| OpenAI 兼容客户端 | `OpenAI(base_url=..., api_key='ollama')` |
| system / user messages | `system_prompt` 定语气；user 拼网页正文 |
| 网页抓取 | `week1.scraper.fetch_website_contents` |
| 本地 Ollama | `MODEL = "llama3.2"`，`OLLAMA_BASE_URL` |

## 怎么跑

1. 确保本机 Ollama 已启动，并已 `ollama pull llama3.2`
2. 确认能从仓库根目录导入 `week1.scraper`（本笔记本用 `sys.path` 指到上级）
3. 从上到下运行单元格；最后一格会摘要 `https://huggingface.co`（可改成你自己的 URL）


In [ ]:
# ========== 环境搭建：把课程仓库加进 import 搜索路径 ==========

# 导入标准库 sys：后面要用 sys.path 改 Python 模块搜索路径
import sys
# 导入标准库 os：拼绝对路径，避免相对路径在不同工作目录下失效
import os

# 把仓库根目录（本文件上三级）加入 sys.path，这样就能 import week1.scraper
sys.path.append(os.path.abspath("../../.."))

# ========== 导入：OpenAI 兼容客户端 + 课程自带网页抓取 ==========

# 从 openai 导入 OpenAI：这里会指向本地 Ollama 的 /v1 兼容接口（不是云端）
from openai import OpenAI
# 从课程 week1.scraper 导入 fetch_website_contents：抓取网页正文文本
from week1.scraper import fetch_website_contents


In [ ]:
# ========== 常量 + Prompt：模型地址与「怎么摘要」的指令集中写这里 ==========

# Ollama 的 OpenAI 兼容基址：本地 11434 端口的 /v1（Chat Completions 形态）
OLLAMA_BASE_URL = "http://localhost:11434/v1"
# 本地模型名：须与本机 `ollama list` 里的名字一致
MODEL = "llama3.2"

# system prompt 保留英文：发给模型的角色/语气指令，翻译会改变行为
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

# user 前缀：真正网页正文会拼在后面；同样保留英文，避免改变模型任务理解
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""


In [ ]:
# ========== messages_for：把网页正文装进 Chat Completions 的 messages 列表 ==========

def messages_for(website):
    """Create message list for the LLM."""
    # 组装 OpenAI 风格 messages 列表，供 chat.completions.create 使用
    return [
        # system：全局规则与吐槽风格（内容来自 system_prompt 常量）
        {"role": "system", "content": system_prompt},
        # user：前缀说明任务 + 抓到的网页正文 website
        {"role": "user", "content": user_prompt_prefix + website}
    ]


In [ ]:
# ========== summarize：抓网页 → 调本地 Ollama → 取出摘要字符串 ==========

def summarize(url):
    """Fetch and summarize a website using Ollama."""
    # 创建指向本地 Ollama 的客户端；api_key 在兼容模式下常填占位 'ollama'
    ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
    # 按 URL 抓取网站正文（课程 scraper 会处理导航噪音等）
    website = fetch_website_contents(url)
    # 非流式 Chat Completions：messages 来自 messages_for(website)
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages_for(website)
    )
    # 从 choices[0].message.content 取出模型生成的 Markdown 摘要
    return response.choices[0].message.content


In [ ]:
# ========== 试跑：对 Hugging Face 首页做一次本地模型摘要 ==========

# 把返回的摘要字符串打印出来；可把 URL 换成你想分析的站点
print(summarize("https://huggingface.co"))
